# Anomaly Detection - Burn-in Data

In [1]:
import json
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Dict, List

import joblib
import numpy as np
import pandas as pd
import shap
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

pd.set_option('display.max_columns', None)

## Config

In [2]:
CSV_PATH = "burn_in_dataset.csv"
SPLIT_PATH = "component_split.csv"

## Shared train/val/test split

In [3]:
def build_component_split(df, batch_col='Batch_ID', label_col='Label', seed=134, block=5):
    batch_fail = df.groupby(batch_col)[label_col].apply(lambda s: (s == 'Fail').mean())
    order = batch_fail.sort_values().index.tolist()
    rng = np.random.RandomState(seed)
    cycle_base = ['train', 'train', 'train', 'val', 'test']
    split_map = {}
    for i in range(0, len(order), block):
        chunk = order[i:i + block]
        if len(chunk) == block:
            roles = cycle_base.copy()
        else:
            roles = list(rng.choice(cycle_base, size=len(chunk), replace=False))
        rng.shuffle(roles)
        for b, r in zip(chunk, roles):
            split_map[b] = r
    out = df[['Component_ID', batch_col]].copy()
    out['split'] = out[batch_col].map(split_map)
    return out

raw_df = pd.read_csv(CSV_PATH)
if Path(SPLIT_PATH).exists():
    split_df = pd.read_csv(SPLIT_PATH)
else:
    split_df = build_component_split(raw_df)
    split_df.to_csv(SPLIT_PATH, index=False)

print(split_df['split'].value_counts())
_check = raw_df.merge(split_df[['Component_ID','split']], on='Component_ID')
print(_check.groupby('split')['Label'].value_counts(normalize=True).unstack().round(3))

split
train    6522
val      1861
test     1617
Name: count, dtype: int64
Label  Borderline   Fail   Safe
split                          
test        0.186  0.165  0.649
train       0.181  0.169  0.650
val         0.192  0.169  0.638


## Pipeline config + validation

In [4]:
@dataclass
class PipelineConfig:
    id_col: str = "Component_ID"
    lot_col: str = "Batch_ID"
    group_col: str = "Part_Type"
    absolute_result_col: str = "Traditional_Test_Result"
    absolute_fail_value: str = "Fail"
    label_col: str = "Label"
    params: List[str] = field(default_factory=lambda: ["Leakage", "Resistance", "Vth"])
    early_timepoints: List[str] = field(default_factory=lambda: ["0h", "24h"])
    weight_lot_relative: float = 0.40
    weight_multivariate: float = 0.40
    weight_drift: float = 0.20
    threshold_monitor: float = 30.0
    threshold_hold: float = 60.0
    threshold_reject: float = 80.0
    z_score_cap: float = 8.0
    iforest_n_estimators: int = 300
    iforest_contamination: str = "auto"
    iforest_min_group_size: int = 30
    random_state: int = 42

    def __post_init__(self):
        total = self.weight_lot_relative + self.weight_multivariate + self.weight_drift
        if abs(total - 1.0) > 1e-6:
            raise ValueError(f"Layer weights must sum to 1.0, got {total:.3f}")
        if not (self.threshold_monitor < self.threshold_hold < self.threshold_reject):
            raise ValueError("Thresholds must be strictly increasing")

    def value_col(self, param, timepoint): return f"{param}_{timepoint}"
    def delta_col(self, param): return f"{param}_delta_24h"
    def pct_change_col(self, param): return f"{param}_pct_change_24h"
    def early_value_cols(self):
        return [self.value_col(p, t) for p in self.params for t in self.early_timepoints]
    def lot_relative_cols(self):
        latest = self.early_timepoints[-1]
        return [self.value_col(p, latest) for p in self.params] + [self.delta_col(p) for p in self.params]
    def multivariate_feature_cols(self):
        cols = []
        for p in self.params:
            cols += [self.value_col(p, t) for t in self.early_timepoints]
            cols += [self.delta_col(p), self.pct_change_col(p)]
        return cols
    def required_input_cols(self):
        return [self.id_col, self.lot_col, self.group_col, self.absolute_result_col] + self.early_value_cols()


class SchemaError(ValueError): pass

def validate_schema(df, config, require_label=False):
    required = list(config.required_input_cols())
    if require_label: required.append(config.label_col)
    missing = [c for c in required if c not in df.columns]
    if missing: raise SchemaError(f"Input data is missing required columns: {missing}")

## Imputation, drift features, the three scoring layers

In [5]:
class LotMedianImputer:
    def __init__(self, config):
        self.config = config; self.columns = config.early_value_cols()
        self.group_medians_ = {}; self.global_medians_ = {}
    def fit(self, df):
        lot_col = self.config.lot_col
        for col in self.columns:
            self.group_medians_[col] = df.groupby(lot_col)[col].median().to_dict()
            self.global_medians_[col] = float(df[col].median())
        return self
    def transform(self, df):
        df = df.copy(); lot_col = self.config.lot_col
        for col in self.columns:
            if df[col].isna().sum() == 0: continue
            fallback = df[lot_col].map(self.group_medians_[col]).fillna(self.global_medians_[col])
            df[col] = df[col].fillna(fallback)
        return df
    def fit_transform(self, df): return self.fit(df).transform(df)


def add_drift_features(df, config):
    df = df.copy(); first_tp, last_tp = config.early_timepoints[0], config.early_timepoints[-1]
    for p in config.params:
        v_first = df[config.value_col(p, first_tp)]; v_last = df[config.value_col(p, last_tp)]
        df[config.delta_col(p)] = v_last - v_first
        df[config.pct_change_col(p)] = np.where(v_first != 0, (v_last - v_first) / v_first.abs(), 0.0)
    return df


class AbsoluteSpecLayer:
    def __init__(self, config): self.config = config
    def transform(self, df):
        col, fail_value = self.config.absolute_result_col, self.config.absolute_fail_value
        return (df[col].astype(str).str.strip().str.lower() == fail_value.lower()).astype(int)


def _robust_median_mad(series):
    median = float(series.median()); mad = float((series - median).abs().median()) * 1.4826
    return median, mad

class LotRelativeScorer:
    def __init__(self, config):
        self.config = config; self.columns = config.lot_relative_cols()
        self.group_stats_ = {}; self.global_stats_ = {}
    def fit(self, df):
        lot_col = self.config.lot_col
        for col in self.columns:
            self.group_stats_[col] = df.groupby(lot_col)[col].apply(_robust_median_mad).to_dict()
            self.global_stats_[col] = _robust_median_mad(df[col])
        return self
    def transform(self, df):
        lot_col = self.config.lot_col
        abs_z_frame = pd.DataFrame(index=df.index)
        for col in self.columns:
            medians = df[lot_col].map(lambda lot: self.group_stats_[col].get(lot, (np.nan, np.nan))[0])
            mads = df[lot_col].map(lambda lot: self.group_stats_[col].get(lot, (np.nan, np.nan))[1])
            g_med, g_mad = self.global_stats_[col]
            medians = medians.fillna(g_med)
            mads = mads.fillna(g_mad).replace(0, g_mad if g_mad > 0 else 1e-6)
            abs_z_frame[col] = ((df[col] - medians) / mads).abs()
        worst_z = abs_z_frame.max(axis=1)
        score = (worst_z.clip(upper=self.config.z_score_cap) / self.config.z_score_cap) * 100
        return score, worst_z


GLOBAL_KEY = "__global__"
class MultivariateAnomalyScorer:
    def __init__(self, config):
        self.config = config; self.feature_cols = config.multivariate_feature_cols()
        self.models_ = {}; self.score_bounds_ = {}
        self.background_ = {}   # FIX: per-group background sample, stored at fit time, for SHAP later

    def _fit_one(self, X):
        pipeline = Pipeline([("scaler", StandardScaler()),
                              ("iforest", IsolationForest(n_estimators=self.config.iforest_n_estimators,
                                                           contamination=self.config.iforest_contamination,
                                                           random_state=self.config.random_state))])
        pipeline.fit(X)
        raw = -pipeline.decision_function(X)
        p1, p99 = np.percentile(raw, [1, 99])
        return pipeline, (float(p1), float(p99))

    def fit(self, df):
        group_col = self.config.group_col
        n_bg = min(100, len(df))
        model, bounds = self._fit_one(df[self.feature_cols].values)
        self.models_[GLOBAL_KEY] = model; self.score_bounds_[GLOBAL_KEY] = bounds
        self.background_[GLOBAL_KEY] = df[self.feature_cols].sample(n_bg, random_state=self.config.random_state)
        for group_value, group_df in df.groupby(group_col):
            if len(group_df) < self.config.iforest_min_group_size:
                print(f"WARNING: group '{group_value}' has only {len(group_df)} rows; using global fallback")
                continue
            model, bounds = self._fit_one(group_df[self.feature_cols].values)
            self.models_[str(group_value)] = model; self.score_bounds_[str(group_value)] = bounds
            n_bg_g = min(100, len(group_df))
            self.background_[str(group_value)] = group_df[self.feature_cols].sample(n_bg_g, random_state=self.config.random_state)
        return self

    def transform(self, df):
        group_col = self.config.group_col
        scores = pd.Series(index=df.index, dtype=float)
        for group_value, group_df in df.groupby(group_col):
            key = str(group_value) if str(group_value) in self.models_ else GLOBAL_KEY
            model = self.models_[key]; p1, p99 = self.score_bounds_[key]
            raw = -model.decision_function(group_df[self.feature_cols].values)
            norm = np.clip((raw - p1) / (p99 - p1 + 1e-9), 0, 1) * 100
            scores.loc[group_df.index] = norm
        return scores

    def shap_top_features(self, df, top_k=3):
        group_col = self.config.group_col
        out = pd.Series(index=df.index, dtype=object)
        for group_value, group_df in df.groupby(group_col):
            key = str(group_value) if str(group_value) in self.models_ else GLOBAL_KEY
            model = self.models_[key]
            background = self.background_[key]
            score_fn = lambda X, _m=model: -_m.decision_function(X)
            explainer = shap.Explainer(score_fn, background.values, feature_names=self.feature_cols)
            sv = explainer(group_df[self.feature_cols].values)
            for i, idx in enumerate(group_df.index):
                vals = np.asarray(sv.values[i]).reshape(-1)
                order = np.argsort(-np.abs(vals))[:top_k]
                out.loc[idx] = ', '.join(f'{self.feature_cols[j]}({vals[j]:+.2f})' for j in order)
        return out


class DriftSeverityScorer:
    def __init__(self, config): self.config = config; self.bound_ = None
    def _severity(self, df):
        cols = [self.config.pct_change_col(p) for p in self.config.params]
        return df[cols].abs().max(axis=1)
    def fit(self, df):
        bound = float(self._severity(df).quantile(0.99)); self.bound_ = bound if bound > 0 else 1e-6
        return self
    def transform(self, df):
        return (self._severity(df).clip(upper=self.bound_) / self.bound_) * 100


def combine(lot_score, mv_score, drift_score, absolute_fail, config):
    combined = (config.weight_lot_relative*lot_score + config.weight_multivariate*mv_score + config.weight_drift*drift_score)
    combined = np.where(absolute_fail == 1, np.maximum(combined, config.threshold_reject + 5), combined)
    return pd.Series(np.round(np.clip(combined, 0, 100), 1), index=lot_score.index)


def decide(risk_score, config):
    bins = [-np.inf, config.threshold_monitor, config.threshold_hold, config.threshold_reject, np.inf]
    return pd.cut(risk_score, bins=bins, labels=["PASS","MONITOR","HOLD","REJECT"], right=False).astype(str)


class AnomalyPipeline:
    def __init__(self, config=None):
        self.config = config or PipelineConfig()
        self.imputer = LotMedianImputer(self.config)
        self.lot_scorer = LotRelativeScorer(self.config)
        self.mv_scorer = MultivariateAnomalyScorer(self.config)
        self.drift_scorer = DriftSeverityScorer(self.config)
        self.absolute_layer = AbsoluteSpecLayer(self.config)

    def fit(self, df):
        validate_schema(df, self.config)
        df = self.imputer.fit_transform(df); df = add_drift_features(df, self.config)
        self.lot_scorer.fit(df); self.mv_scorer.fit(df); self.drift_scorer.fit(df)
        return self

    def predict(self, df):
        validate_schema(df, self.config)
        original_index = df.index
        df = self.imputer.transform(df); df = add_drift_features(df, self.config)
        absolute_fail = self.absolute_layer.transform(df)
        lot_score, worst_z = self.lot_scorer.transform(df)
        mv_score = self.mv_scorer.transform(df)
        drift_score = self.drift_scorer.transform(df)
        risk_score = combine(lot_score, mv_score, drift_score, absolute_fail, self.config)
        decision = decide(risk_score, self.config)
        result = pd.DataFrame({
            self.config.id_col: df[self.config.id_col], self.config.lot_col: df[self.config.lot_col],
            self.config.group_col: df[self.config.group_col], "Absolute_Spec_Fail": absolute_fail,
            "Lot_Relative_Score": lot_score.round(1), "Worst_Lot_Zscore": worst_z.round(2),
            "Multivariate_Score": mv_score.round(1), "Drift_Score": drift_score.round(1),
            "Anomaly_Risk_Score": risk_score, "QA_Decision": decision,
        }, index=original_index)
        if self.config.label_col in df.columns: result[self.config.label_col] = df[self.config.label_col]
        if self.config.absolute_result_col in df.columns: result[self.config.absolute_result_col] = df[self.config.absolute_result_col]
        return result

## Fit on `train` only


In [6]:
df = raw_df.merge(split_df[['Component_ID','split']], on='Component_ID')
df_train = df[df['split']=='train'].reset_index(drop=True)
df_val   = df[df['split']=='val'].reset_index(drop=True)
df_test  = df[df['split']=='test'].reset_index(drop=True)

base_config = PipelineConfig()
pipeline = AnomalyPipeline(base_config)
pipeline.fit(df_train)
print(f'fit on {len(df_train)} train rows')

fit on 6522 train rows


## Weights and thresholds are chosen to hit a target recall, not just AUC


In [7]:
def raw_layer_scores(fitted_pipeline, cfg, df_split):
    dfx = fitted_pipeline.imputer.transform(df_split)
    dfx = add_drift_features(dfx, cfg)
    absolute_fail = fitted_pipeline.absolute_layer.transform(dfx)
    lot_score, worst_z = fitted_pipeline.lot_scorer.transform(dfx)
    mv_score = fitted_pipeline.mv_scorer.transform(dfx)
    drift_score = fitted_pipeline.drift_scorer.transform(dfx)
    return lot_score, mv_score, drift_score, absolute_fail

lot_v, mv_v, drift_v, absfail_v = raw_layer_scores(pipeline, base_config, df_val)
label_v = df_val['Label'].values

TARGET_RECALL = 0.98

def threshold_for_recall(combined, label, target_recall):
    fail_mask = (label == 'Fail')
    n_fail = int(fail_mask.sum())
    if n_fail == 0:
        return None
    order = np.argsort(-combined)  # most suspicious first
    caught = 0
    for idx in order:
        if fail_mask[idx]:
            caught += 1
        if caught / n_fail >= target_recall:
            thr = combined[idx]
            flagged = combined >= thr
            recall = float((flagged & fail_mask).sum()) / n_fail
            safe_mask = (label == 'Safe')
            fa = float((flagged & safe_mask).sum()) / max(int(safe_mask.sum()), 1)
            return float(thr), recall, fa
    return None

rng = np.random.RandomState(0)
best = None  # (false_alarm_rate, -auc, weights, threshold_hold, achieved_recall, auc)
for _ in range(400):
    w = rng.dirichlet([1,1,1])
    combined = w[0]*lot_v + w[1]*mv_v + w[2]*drift_v
    combined = np.where(absfail_v==1, 100.0, combined)
    result = threshold_for_recall(combined, label_v, TARGET_RECALL)
    if result is None:
        continue
    thr_hold, recall, fa = result
    auc = roc_auc_score((label_v=='Fail').astype(int), combined)
    key = (fa, -auc)
    if best is None or key < best[0]:
        best = (key, w, thr_hold, recall, fa, auc)

if best is None:
    raise RuntimeError(
        f'No weight combination reached {TARGET_RECALL:.0%} recall on val Fail cases with the '
        'current features - lower TARGET_RECALL or add features/data before shipping this tier.'
    )

_, best_w, thr_hold, achieved_recall, achieved_fa, best_auc = best
print(f'selected weights (lot, multivariate, drift): {best_w.round(3)}')
print(f'-> {achieved_recall:.1%} recall on val Fail via HOLD/REJECT, at {achieved_fa:.1%} false-alarm rate on val Safe '
      f'(AUC={best_auc:.4f}, reported for context only)')

combined_v = best_w[0]*lot_v + best_w[1]*mv_v + best_w[2]*drift_v
combined_v = np.where(absfail_v==1, 100.0, combined_v)
safe_scores = combined_v[label_v=='Safe']

thr_monitor = min(float(np.percentile(safe_scores, 85)), thr_hold - 0.1)
thr_reject = max(float(np.percentile(safe_scores, 99.5)), thr_hold + 0.1)

tuned_config = PipelineConfig(
    weight_lot_relative=round(float(best_w[0]),3),
    weight_multivariate=round(float(best_w[1]),3),
    weight_drift=round(1 - round(float(best_w[0]),3) - round(float(best_w[1]),3), 3),
    threshold_monitor=round(thr_monitor,1), threshold_hold=round(thr_hold,1), threshold_reject=round(thr_reject,1),
)
print(tuned_config)

selected weights (lot, multivariate, drift): [0.28  0.267 0.453]
-> 98.1% recall on val Fail via HOLD/REJECT, at 40.8% false-alarm rate on val Safe (AUC=0.8882, reported for context only)
PipelineConfig(id_col='Component_ID', lot_col='Batch_ID', group_col='Part_Type', absolute_result_col='Traditional_Test_Result', absolute_fail_value='Fail', label_col='Label', params=['Leakage', 'Resistance', 'Vth'], early_timepoints=['0h', '24h'], weight_lot_relative=0.28, weight_multivariate=0.267, weight_drift=0.453, threshold_monitor=13.9, threshold_hold=14.0, threshold_reject=100.0, z_score_cap=8.0, iforest_n_estimators=300, iforest_contamination='auto', iforest_min_group_size=30, random_state=42)


## Report train / val / test separately - test is the honest number


In [8]:
final_pipeline = AnomalyPipeline(tuned_config)
final_pipeline.fit(df_train)

train_results = final_pipeline.predict(df_train); train_results['split']='train'
val_results   = final_pipeline.predict(df_val);   val_results['split']='val'
test_results  = final_pipeline.predict(df_test);  test_results['split']='test'
all_results = pd.concat([train_results, val_results, test_results], ignore_index=True)

def generate_report(results, name):
    print(f'--- {name} (n={len(results)}) ---')
    counts = results['QA_Decision'].value_counts().reindex(['PASS','MONITOR','HOLD','REJECT']).fillna(0).astype(int)
    for k,v in counts.items():
        print(f'  {k:<10} {v:>6,} ({v/len(results):.1%})')
    fail_mask = results['Label']=='Fail'
    caught = results.loc[fail_mask,'QA_Decision'].isin(['HOLD','REJECT']).mean() if fail_mask.any() else float('nan')
    flag = '' if not fail_mask.any() or caught >= TARGET_RECALL else f'  <-- BELOW TARGET_RECALL ({TARGET_RECALL:.0%})'
    print(f'  Recall on true FAIL (HOLD/REJECT): {caught:.1%}{flag}')
    safe_mask = results['Label']=='Safe'
    false_alarm = results.loc[safe_mask,'QA_Decision'].isin(['HOLD','REJECT']).mean()
    print(f'  False-alarm rate on true SAFE (HOLD/REJECT): {false_alarm:.1%}')
    print()

generate_report(train_results, 'TRAIN (in-sample, optimistic - reference only')
generate_report(val_results,   'VAL (used to tune weights/thresholds - recall was tuned to hit TARGET_RECALL here)')
generate_report(test_results,  'TEST (held-out - the number to actually trust)')

--- TRAIN (in-sample, optimistic - reference only (n=6522) ---
  PASS        2,561 (39.3%)
  MONITOR        19 (0.3%)
  HOLD        2,716 (41.6%)
  REJECT      1,226 (18.8%)
  Recall on true FAIL (HOLD/REJECT): 96.6%  <-- BELOW TARGET_RECALL (98%)
  False-alarm rate on true SAFE (HOLD/REJECT): 42.4%

--- VAL (used to tune weights/thresholds - recall was tuned to hit TARGET_RECALL here) (n=1861) ---
  PASS          716 (38.5%)
  MONITOR        11 (0.6%)
  HOLD          838 (45.0%)
  REJECT        296 (15.9%)
  Recall on true FAIL (HOLD/REJECT): 98.1%
  False-alarm rate on true SAFE (HOLD/REJECT): 41.1%

--- TEST (held-out - the number to actually trust) (n=1617) ---
  PASS          607 (37.5%)
  MONITOR         4 (0.2%)
  HOLD          669 (41.4%)
  REJECT        337 (20.8%)
  Recall on true FAIL (HOLD/REJECT): 99.3%
  False-alarm rate on true SAFE (HOLD/REJECT): 44.4%



## SHAP explanations for flagged components

In [9]:
flagged_test_mask = test_results['QA_Decision'].isin(['HOLD', 'REJECT'])
flagged_test_ids = set(test_results.loc[flagged_test_mask, 'Component_ID'])

# Need the (imputed, drift-featurized) feature frame, not the raw input, to feed the scorer
dfx_test = final_pipeline.imputer.transform(df_test)
dfx_test = add_drift_features(dfx_test, tuned_config)
dfx_test_flagged = dfx_test[dfx_test['Component_ID'].isin(flagged_test_ids)]

shap_explanations = final_pipeline.mv_scorer.shap_top_features(dfx_test_flagged, top_k=3)
shap_by_id = dict(zip(dfx_test_flagged['Component_ID'], shap_explanations.values))

all_results['SHAP_Top_Anomaly_Features'] = all_results['Component_ID'].map(shap_by_id)
print(f'SHAP attributions computed for {len(shap_explanations)} flagged TEST components')
all_results.loc[all_results['split']=='test', ['Component_ID','QA_Decision','SHAP_Top_Anomaly_Features']].dropna(subset=['SHAP_Top_Anomaly_Features']).head(10)

PermutationExplainer explainer: 507it [09:19,  1.11s/it]
PermutationExplainer explainer: 501it [08:47,  1.07s/it]

SHAP attributions computed for 1006 flagged TEST components


,Component_ID,QA_Decision,SHAP_Top_Anomaly_Features
8383,C505,REJECT,"Resistance_0h(+0.01), Leakage_0h(-0.01), Leaka..."
8385,C523,HOLD,"Vth_0h(-0.01), Resistance_0h(+0.00), Leakage_d..."
8388,C541,HOLD,"Leakage_delta_24h(+0.01), Vth_24h(-0.01), Resi..."
8389,C544,HOLD,"Leakage_0h(-0.01), Vth_24h(+0.01), Resistance_..."
8391,C561,HOLD,"Vth_0h(+0.03), Vth_24h(+0.02), Leakage_delta_2..."
8393,C578,REJECT,"Vth_0h(+0.02), Resistance_0h(+0.02), Vth_24h(+..."
8394,C586,REJECT,"Leakage_delta_24h(+0.01), Resistance_pct_chang..."
8398,C621,HOLD,"Resistance_delta_24h(+0.01), Resistance_pct_ch..."
8399,C625,HOLD,"Vth_0h(+0.02), Resistance_0h(+0.02), Vth_24h(+..."
8401,C639,HOLD,"Vth_0h(-0.01), Leakage_0h(-0.01), Vth_24h(-0.00)"


## Reuse on new data

In [10]:
new_batch = df_test.sample(20, random_state=7)
new_batch_scored = final_pipeline.predict(new_batch)
new_batch_scored.head(20)

,Component_ID,Batch_ID,Part_Type,Absolute_Spec_Fail,Lot_Relative_Score,Worst_Lot_Zscore,Multivariate_Score,Drift_Score,Anomaly_Risk_Score,QA_Decision,Label,Traditional_Test_Result
530,C3836,ISRO-SCL-250120-03,Op-Amp IC,0,15.0,1.20,9.9,26.8,18.9,HOLD,Borderline,Pass
755,C5261,ISRO-SCL-250331-13,Voltage Regulator IC,1,17.8,1.42,29.3,29.8,100.0,REJECT,Borderline,Fail
140,C1359,ISRO-SCL-250505-18,Op-Amp IC,0,15.7,1.25,9.2,4.5,8.9,PASS,Safe,Pass
1496,C9734,ISRO-SCL-250120-03,Op-Amp IC,0,5.7,0.45,1.5,9.2,6.2,PASS,Safe,Pass
715,C4981,ISRO-SCL-250505-18,Op-Amp IC,0,17.7,1.42,34.9,5.6,16.8,HOLD,Safe,Pass
970,C6477,ISRO-SCL-250331-13,Voltage Regulator IC,0,9.6,0.77,0.0,6.3,5.5,PASS,Safe,Pass
1121,C7462,ISRO-SCL-250331-13,Voltage Regulator IC,0,22.9,1.83,24.8,10.5,17.8,HOLD,Safe,Pass
511,C3689,ISRO-SCL-250331-13,Voltage Regulator IC,0,25.4,2.03,7.3,32.1,23.6,HOLD,Borderline,Pass
699,C4867,ISRO-SCL-250331-13,Voltage Regulator IC,0,15.1,1.21,11.7,24.2,18.3,HOLD,Borderline,Pass
473,C3479,ISRO-SCL-250519-20,Voltage Regulator IC,0,15.7,1.26,5.5,8.3,9.6,PASS,Safe,Pass


## Save

In [11]:
all_results.to_csv('anomaly_results.csv', index=False)
joblib.dump(final_pipeline, 'anomaly_pipeline.joblib')

anomaly_shap_bundle = {
    'iforest_models': final_pipeline.mv_scorer.models_,           # dict[str, sklearn Pipeline] - safe
    'score_bounds': final_pipeline.mv_scorer.score_bounds_,       # dict[str, (p1, p99)]
    'background': final_pipeline.mv_scorer.background_,           # dict[str, pd.DataFrame]
    'feature_cols': final_pipeline.mv_scorer.feature_cols,        # list[str]
    'group_col': tuned_config.group_col,                          # 'Part_Type'
    'lot_col': tuned_config.lot_col,                              # 'Batch_ID'
    'params': tuned_config.params,                                # ['Leakage','Resistance','Vth']
    'early_timepoints': tuned_config.early_timepoints,            # ['0h','24h']
    'early_value_cols': tuned_config.early_value_cols(),          # for replicating imputation
    'group_medians': final_pipeline.imputer.group_medians_,       # dict[col][lot] -> median
    'global_medians': final_pipeline.imputer.global_medians_,     # dict[col] -> median
}
joblib.dump(anomaly_shap_bundle, 'anomaly_shap_bundle.joblib')
print('saved anomaly_results.csv, anomaly_pipeline.joblib (session-local), and anomaly_shap_bundle.joblib (portable, for Module C)')

saved anomaly_results.csv, anomaly_pipeline.joblib (session-local), and anomaly_shap_bundle.joblib (portable, for Module C)
